# Benchmark: Base-Draft Speculative Verification & vLLM `WSI_SPEC` (`Qwen/Qwen3.5-0.8B`, Colab L4)

Evalúa las dos piezas de aceleración para evaluación de parches WSI sobre GSM8K:
1. **PyTorch (`wsi.spec_eval`)**: usa los borradores cacheados del modelo `base` (+ re-alineamiento con `SuffixDraftIndex`) para verificar en bloques de 16 tokens sobre el modelo con `patches/qwen35_0.8b_p50c_loop_fix.moipatch` (`always-on`), comprobando coincidencia **bit a bit** con `greedy_generate_reference`, `tokens_per_forward`, `acceptance_rate` y *speedup* real.
2. **vLLM `WSI_SPEC` (`none`, `ngram`, `suffix`, `mtp`)**: ejecuta `base` + 2 configuraciones parcheadas *in-memory* por cada modo con `loop_kill_v2` activo y compara `tok/s`, arranque y paridad de exactitud/bucles.

Guarda el JSON final de forma atómica en `MyDrive/wsi_runs/FINALES/bench_base_draft_spec_l4.json` y apaga el runtime solo si el archivo quedó verificado en Drive.

In [ ]:
MODEL = "Qwen/Qwen3.5-0.8B"
RUN_NAME = "bench_base_draft_spec_l4"
REPO = "/content/wsi"
BASE = "/content/base"
PYTORCH_N = 16
PYTORCH_MAX_TOK = 512
VLLM_N = 32
VLLM_MAX_TOK = 2048
BLOCK_SIZE = 16

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm pandas pyarrow huggingface_hub safetensors transformers hf_transfer
!pip install -q flash-linear-attention || echo "fla opcional"
!pip install -q arctic-inference || echo "arctic-inference opcional (suffix se saltará si falta)"
import torch, transformers, vllm
print("torch", torch.__version__, "| transformers", transformers.__version__, "| vllm", vllm.__version__)

In [ ]:
import os, json, subprocess, datetime
from google.colab import drive
drive.mount("/content/drive")
FINALES = "/content/drive/MyDrive/wsi_runs/FINALES"
VLLM_CACHE = "/content/drive/MyDrive/wsi_runs/_vllm_cache"
os.makedirs(FINALES, exist_ok=True)
os.makedirs(VLLM_CACHE, exist_ok=True)
os.environ["VLLM_CACHE_ROOT"] = VLLM_CACHE
os.environ["FLASHINFER_WORKSPACE_BASE"] = "/content/flashinfer_ws"
os.makedirs("/content/flashinfer_ws", exist_ok=True)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
else:
    !cd {REPO} && git pull -q
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()

from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit:", COMMIT, "| modelo en:", BASE)

In [ ]:
import os, signal, subprocess, sys
my_pid = os.getpid()
pids = subprocess.check_output(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"], text=True).splitlines()
for raw in pids:
    raw = raw.strip()
    if raw.isdigit() and int(raw) != my_pid:
        try:
            os.kill(int(raw), signal.SIGKILL)
        except OSError:
            pass

LOCAL_OUT = f"/content/{RUN_NAME}.json"
PATCH = f"{REPO}/patches/qwen35_0.8b_p50c_loop_fix.moipatch"
SCRIPT = f"{REPO}/scripts/bench_base_draft_spec.py"
if not os.path.exists(SCRIPT):
    raise FileNotFoundError(f"Falta {SCRIPT} en el clon {REPO} (commit {COMMIT}). Hay que hacer commit+push a GitHub antes de ejecutar en Colab.")

cmd = [
    sys.executable, "-u", SCRIPT,
    "--model-dir", BASE,
    "--patch", PATCH,
    "--mode", "both",
    "--n", str(PYTORCH_N), "--max-tokens", str(PYTORCH_MAX_TOK), "--block-size", str(BLOCK_SIZE), "--include-sampling",
    "--vllm-n", str(VLLM_N), "--vllm-max-tokens", str(VLLM_MAX_TOK),
    "--out", LOCAL_OUT,
]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

In [ ]:
res = json.load(open(LOCAL_OUT, encoding="utf-8"))
print("=== PYTORCH BASE-DRAFT SPECULATIVE EVAL ===")
for k, a in res.get("pytorch", {}).get("arms", {}).items():
    print(f"{k:20s} | tok/fwd={a.get('tokens_per_forward', '-'):>6} | acc_rate={a.get('acceptance_rate', '-'):>6} "
          f"| exact={a.get('exact_match_frac', '-'):>6} | tok/s={a.get('tok_per_s', '-'):>6} | wall={a.get('elapsed_s')}s")

print("\n=== vLLM WSI_SPEC SUMMARY ===")
for r in res.get("vllm", []):
    if r.get("status") == "ok":
        print(f"{r['mode']:8s} | OK | tok/s={r['tok_per_s']:>6.1f} | gen={r['gen_s']:>5.1f}s | init={r['init_s']:>5.1f}s | early_stop={r['early_loop_stop']}")
        for arm_name, av in r.get("arms", {}).items():
            print(f"   {arm_name:10s}: acc={av['accuracy']:.1%} loop_killed={av['loop_killed']:.1%} tok/s={av['tok_per_s']:.1f}")
    else:
        print(f"{r['mode']:8s} | {r.get('status')} | {r.get('reason') or r.get('error', '')[:120]}")

res["_final"] = {
    "complete": True,
    "model": MODEL,
    "commit": COMMIT,
    "notebook": "base_draft_spec_colab.ipynb",
    "gpu": subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip(),
    "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
}
dst = f"{FINALES}/{RUN_NAME}.json"
with open(dst + ".tmp", "w", encoding="utf-8") as f:
    json.dump(res, f, indent=2)
os.replace(dst + ".tmp", dst)
print("\nGuardado final en Drive:", dst)

In [ ]:
# Apagar runtime solo si el JSON final quedó guardado y verificado en Drive
AUTO_APAGAR = True
import time
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst, encoding="utf-8")).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    time.sleep(60)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")